### Packages

In [ ]:
import numpy as np
import matplotlib as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction import text
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

### Load Dataset

In [ ]:
df = pd.read_csv("news_headlines_unlabeled.csv")

print(df.shape)

print(df.head(5))

### Preprocess Text/Form term-document matrix
instead of just creating a term document matrix that looks at word frequencies per document, lets use tfidf to emphasize important words and ignore noise

In [ ]:
# Grab the text column as strings
headlines = df["headline"].astype(str).fillna("").tolist()
news_stop = {
    "says","said","new","live","update","breaking","report","reports",
    "amid","after","first","latest","day","week","year"
}
stop_words = list(text.ENGLISH_STOP_WORDS.union(news_stop))
# TF-IDF (good defaults for short texts like headlines)
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words=stop_words,
    ngram_range=(1,2),
    min_df=3, # minimum number of documents that a term must appear in to be considered a term
    max_df=0.9, # if a term appears in more than that percent of documents, it will not be considered
    sublinear_tf=True, # replaces raw term frequency with 1 + log (freq)
    norm=None, # no normalization factor
    smooth_idf=True, 
    use_idf=True, 
    max_features=10000
)
tfidf_headlines = tfidf.fit_transform(headlines)

print("TF-IDF matrix shape:", tfidf_headlines.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))

### Create SVD class

In [ ]:

def make_svd(n_comp): 
    svd = TruncatedSVD(n_components=n_comp, n_iter=15, random_state=0)
    lsa = make_pipeline(svd, Normalizer(copy=False)) # use normalization because we want to know if documents fit categories, not how much they fit categories
    Z = lsa.fit_transform(tfidf_headlines)  
    print("Explained variance (sum):", svd.explained_variance_ratio_.sum())
    return Z

### Fit SVD

In [ ]:
Z = make_svd(20)

### Visualizations

In [ ]:

# Cluster the document embeddings
kmeans = KMeans(n_clusters=5, init="k-means++", n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(Z)

# Project embeddings to 2D with PCA
pca = PCA(n_components=2, random_state=42)
Z_2d = pca.fit_transform(Z)

# Scatter plot colored by KMeans cluster
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    Z_2d[:, 0],
    Z_2d[:, 1],
    c=cluster_labels,
    cmap="tab10",
    s=18,
    alpha=0.75,
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA projection of headline embeddings colored by kmeans clusters")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()


### Create submission 

In [ ]:

z_cols = [f"z{i}" for i in range(1, Z.shape[1] + 1)]
submission = pd.DataFrame(Z, columns=z_cols)
submission.insert(0, "doc_id", df["doc_id"].astype(int).to_numpy())

submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission.head())